# Loan Approval Prediction Using a Decision Tree

This notebook explores the loan application dataset, prepares features, trains a Decision Tree classification pipeline, and evaluates it on a stratified hold-out test set.

**Target mapping:** `N` = Rejected (0), `Y` = Approved (1). The model is for educational purposes only and must not be used as the sole basis for real lending decisions.

**Dataset:** Place `loan_approval_dataset.csv` in the project root before running the notebook, or update the CSV path in the loading cell.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import joblib
import os

In [ ]:
df = pd.read_csv("../dataset/loan_approval_dataset.csv")

In [ ]:
df

In [ ]:
df.head()

In [ ]:
import os

print(os.getcwd())

In [ ]:
# Display the column names
print(df.columns.tolist())

# Check data types and non-null counts
df.info()

# Statistical summary of numerical columns
display(df.describe())

# Statistical summary of categorical columns
display(df.describe(include="object"))

In [ ]:
# Count missing values in each column
print("Missing values:")
print(df.isnull().sum())

# Count duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check duplicate loan IDs
print("Duplicate Loan IDs:", df["Loan_ID"].duplicated().sum())

In [ ]:
print(df["Loan_Status"].value_counts())

print("\nTarget percentages:")
print(
    df["Loan_Status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

sns.countplot(data=df, x="Loan_Status")
plt.title("Loan Approval Status Distribution")
plt.xlabel("Loan Status")
plt.ylabel("Number of Applicants")
plt.show()

In [ ]:
# Remove the identifier and excluded personal attributes
X = df.drop(
    columns=["Loan_ID", "Gender", "Married", "Loan_Status"]
)

# Convert target labels into numbers
y = df["Loan_Status"].str.strip().map({
    "N": 0,
    "Y": 1
})

# Validate target conversion
assert y.notna().all(), "Unexpected Loan_Status values found!"

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget values:")
print(y.value_counts())

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numeric_features
        )
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

In [ ]:
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                criterion="gini",
                max_depth=5,
                min_samples_split=10,
                min_samples_leaf=5,
                random_state=42
            )
        )
    ]
)

# The trained classifier is available as model_pipeline.named_steps['classifier'].
# No separate, untrained DecisionTreeClassifier is needed here.

# The trained classifier is available as model_pipeline.named_steps['classifier'].
# No separate, untrained DecisionTreeClassifier is needed here.

In [ ]:

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# Get the trained Decision Tree from your pipeline
tree_model = model_pipeline.named_steps["classifier"]

# Get feature names after categorical encoding
feature_names = (
    model_pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Plot the actual trained tree
plt.figure(figsize=(28, 14), dpi=150)

plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=["Rejected", "Approved"],
    filled=True,
    rounded=True,
    proportion=False,
    precision=2,
    fontsize=8,
    max_depth=4
)

plt.title(
    "Loan Approval Prediction - Decision Tree",
    fontsize=18,
    pad=20
)

plt.tight_layout()
plt.show()


In [ ]:
model_pipeline.fit(X_train, y_train)

print("Decision Tree training completed!")

In [ ]:
y_train_pred = model_pipeline.predict(X_train)
y_test_pred = model_pipeline.predict(X_test)

print("First 10 actual labels:")
print(y_test.head(10).tolist())

print("\nFirst 10 predicted labels:")
print(y_test_pred[:10].tolist())

In [ ]:
train_accuracy = accuracy_score(
    y_train,
    y_train_pred
)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")

In [ ]:
print(
    classification_report(
        y_test,
        y_test_pred,
        labels=[0, 1],
        target_names=["Rejected", "Approved"],
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=[0, 1]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Rejected", "Approved"]
)

disp.plot(cmap="Blues")
plt.title("Decision Tree Confusion Matrix")
plt.show()

In [ ]:
from sklearn.tree import export_text

# Get the trained Decision Tree
tree_model = model_pipeline.named_steps["classifier"]

# Get the feature names after preprocessing
feature_names = (
    model_pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Print the decision tree rules
tree_rules = export_text(
    tree_model,
    feature_names=list(feature_names)
)

print(tree_rules)